In [5]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('infix.xlsx')

df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))
df['Discount_Number'] = df['Discount'].apply(extract_discount)

df.drop("Discount", axis=1, inplace=True)

brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai+', 'ai', 'aiplus', 'ai plus', 'a i'], 
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'redmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infinix': ['Infinix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)


df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)


def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')


def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [6]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,Infinix GT 30 5G+ Pulse Green 128 (Pulse Green...,18999.0,24999.0,4.4,"1,012 Ratings & 58 Reviews",8,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,64MP + 8MP | 13MP Front Camera,5500,Dimensity 7400 Processor,https://www.flipkart.com/infinix-gt-30-5g-puls...,https://rukminim2.flixcart.com/image/312/312/x...,24,Unknown,1012.0,58.0
1,Infinix GT 30 5G+ Pulse Green 256 (Pulse Green...,20999.0,26999.0,4.4,377 Ratings & 29 Reviews,8,256,17.22 cm (6.78 inch) Display,64MP + 8MP | 13MP Front Camera,5500,Dimensity 7400 Processor,https://www.flipkart.com/infinix-gt-30-5g-puls...,https://rukminim2.flixcart.com/image/312/312/x...,22,Unknown,377.0,29.0
2,"Infinix SMART 10 (Iris Blue, 64 GB)",10999.0,11999.0,4.3,"1,200 Ratings & 76 Reviews",4,64,16.94 cm (6.67 inch) HD+ Display,8MP Rear Camera | 8MP Front Camera,5000,UniSoc T7250 Processor,https://www.flipkart.com/infinix-smart-10-iris...,https://rukminim2.flixcart.com/image/312/312/x...,8,Unknown,1200.0,76.0
3,"Infinix GT 30 5G+ Cyber Blue 256 (Cyber Blue, ...",20999.0,26999.0,4.5,345 Ratings & 32 Reviews,8,256,17.22 cm (6.78 inch) Display,64MP + 8MP | 13MP Front Camera,5500,Dimensity 7400 Processor,https://www.flipkart.com/infinix-gt-30-5g-cybe...,https://rukminim2.flixcart.com/image/312/312/x...,22,Unknown,345.0,32.0
4,"Infinix Hot 60 5G+ (Caramel Glow, 128 GB)",17999.0,NaN,4.3,"2,795 Ratings & 170 Reviews",6,128,17.02 cm (6.7 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,5200,Dimensity 7020 Processor,https://www.flipkart.com/infinix-hot-60-5g-car...,https://rukminim2.flixcart.com/image/312/312/x...,0,Unknown,2795.0,170.0


In [7]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                58
Rating             10
Review text        10
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor          10
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings            10
Reviews            10
dtype: int64

In [8]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [9]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                 0
Rating             10
Review text        10
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor          10
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings            10
Reviews            10
dtype: int64

In [10]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Ratings'] = df['Ratings'].fillna(df['Ratings'].median()).astype(int)
df['Reviews'] = df['Reviews'].fillna(df['Reviews'].median()).astype(int)

In [11]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                 0
Rating              0
Review text        10
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor          10
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             0
Reviews             0
dtype: int64

In [12]:
def get_infinix_series(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    # Zero Series (Flagship)
    if 'Zero 40' in product_name:
        if 'Pro' in product_name:
            return 'Zero 40 Pro'
        else:
            return 'Zero 40'
    elif 'Zero 30' in product_name:
        if 'Pro' in product_name:
            return 'Zero 30 Pro'
        else:
            return 'Zero 30'
    elif 'Zero 20' in product_name:
        return 'Zero 20'
    elif 'Zero' in product_name:
        return 'Zero Series'
    
    # GT Series (Performance/Gaming)
    elif 'GT 20' in product_name:
        if 'Pro' in product_name:
            return 'GT 20 Pro'
        else:
            return 'GT 20'
    elif 'GT 10' in product_name:
        if 'Pro' in product_name:
            return 'GT 10 Pro'
        else:
            return 'GT 10'
    elif 'GT' in product_name:
        return 'GT Series'
    
    # Note Series (Mid-range)
    elif 'Note 40' in product_name:
        if 'Pro+' in product_name:
            return 'Note 40 Pro+'
        elif 'Pro' in product_name:
            return 'Note 40 Pro'
        else:
            return 'Note 40'
    elif 'Note 30' in product_name:
        if 'Pro' in product_name:
            return 'Note 30 Pro'
        else:
            return 'Note 30'
    elif 'Note 20' in product_name:
        if 'Pro' in product_name:
            return 'Note 20 Pro'
        else:
            return 'Note 20'
    elif 'Note 12' in product_name:
        return 'Note 12'
    elif 'Note 11' in product_name:
        return 'Note 11'
    elif 'Note 10' in product_name:
        return 'Note 10'
    elif 'Note' in product_name:
        return 'Note Series'
    
    # Hot Series (Budget)
    elif 'Hot 40' in product_name:
        if 'Pro' in product_name:
            return 'Hot 40 Pro'
        else:
            return 'Hot 40'
    elif 'Hot 30' in product_name:
        if 'Pro' in product_name:
            return 'Hot 30 Pro'
        else:
            return 'Hot 30'
    elif 'Hot 20' in product_name:
        return 'Hot 20'
    elif 'Hot 12' in product_name:
        return 'Hot 12'
    elif 'Hot 11' in product_name:
        return 'Hot 11'
    elif 'Hot 10' in product_name:
        return 'Hot 10'
    elif 'Hot' in product_name:
        return 'Hot Series'
    
    # Smart Series (Entry-level)
    elif 'Smart 8' in product_name:
        return 'Smart 8'
    elif 'Smart 7' in product_name:
        return 'Smart 7'
    elif 'Smart 6' in product_name:
        return 'Smart 6'
    elif 'Smart' in product_name:
        return 'Smart Series'
    
    else:
        return 'Other'

df['Infinix_Series'] = df['Product Name'].apply(get_infinix_series)


def get_infinix_processor(product_name):
    if pd.isna(product_name):
        return 'Unknown Processor'
    
    # Zero Series
    if 'Zero 40 Pro' in product_name:
        return 'MediaTek Dimensity 8200'
    elif 'Zero 40' in product_name:
        return 'MediaTek Dimensity 8020'
    elif 'Zero 30 Pro' in product_name:
        return 'MediaTek Dimensity 8050'
    elif 'Zero 30' in product_name:
        return 'MediaTek Dimensity 8020'
    elif 'Zero 20' in product_name:
        return 'MediaTek Dimensity 930'
    
    # GT Series
    elif 'GT 20 Pro' in product_name:
        return 'MediaTek Dimensity 8200'
    elif 'GT 20' in product_name:
        return 'MediaTek Dimensity 8200'
    elif 'GT 10 Pro' in product_name:
        return 'MediaTek Dimensity 8050'
    elif 'GT 10' in product_name:
        return 'MediaTek Dimensity 1300'
    
    # Note Series
    elif 'Note 40 Pro+' in product_name:
        return 'MediaTek Dimensity 7020'
    elif 'Note 40 Pro' in product_name:
        return 'MediaTek Dimensity 7020'
    elif 'Note 40' in product_name:
        return 'MediaTek Helio G99'
    elif 'Note 30 Pro' in product_name:
        return 'MediaTek Helio G99'
    elif 'Note 30' in product_name:
        return 'MediaTek Helio G96'
    elif 'Note 20 Pro' in product_name:
        return 'MediaTek Helio G96'
    elif 'Note 20' in product_name:
        return 'MediaTek Helio G85'
    elif 'Note 12' in product_name:
        return 'MediaTek Helio G96'
    elif 'Note 11' in product_name:
        return 'MediaTek Helio G88'
    elif 'Note 10' in product_name:
        return 'MediaTek Helio G85'
    
    # Hot Series
    elif 'Hot 40 Pro' in product_name:
        return 'MediaTek Helio G99'
    elif 'Hot 40' in product_name:
        return 'MediaTek Helio G88'
    elif 'Hot 30 Pro' in product_name:
        return 'MediaTek Helio G99'
    elif 'Hot 30' in product_name:
        return 'MediaTek Helio G88'
    elif 'Hot 20' in product_name:
        return 'MediaTek Helio G85'
    elif 'Hot 12' in product_name:
        return 'MediaTek Helio G85'
    elif 'Hot 11' in product_name:
        return 'MediaTek Helio G70'
    elif 'Hot 10' in product_name:
        return 'MediaTek Helio G70'
    
    # Smart Series
    elif 'Smart 8' in product_name:
        return 'MediaTek Helio G36'
    elif 'Smart 7' in product_name:
        return 'MediaTek Helio G37'
    elif 'Smart 6' in product_name:
        return 'MediaTek Helio G35'
    
    else:
        return 'Unknown Processor'

df.loc[df['Processor'].isnull(), 'Processor'] = df.loc[df['Processor'].isnull(), 'Product Name'].apply(get_infinix_processor)

In [13]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                 0
Rating              0
Review text        10
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor           0
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             0
Reviews             0
Infinix_Series      0
dtype: int64

In [14]:
df.drop("Infinix_Series", axis=1, inplace=True)

In [15]:
df.to_excel('Cleaned_infinix.xlsx', index=False)